# Auditoría: Flujo `empresa` — scraping → limpieza → normalización → DB → API

**Fecha**: 2026-04-29  
**Endpoint**: `GET /api/v1/stats/empresa`  
**Objetivo**: Auditar el flujo completo de la columna `empresa` (nombre de empresa en `ofertas` vía JOIN con `empresas`), detectar fragilidades en limpieza, normalización, modelo y métricas.

---

## Trazado del flujo

```
1. Scraping (extraction.py:54)
   → get_safe_text(offer, "p.dFlex a", default="-")
   → Si no encuentra el nombre: "-"

2. Pipeline (pipeline.py:14-16)
   → sanitize_text(df)           — Unicode + colapso de espacios
   → handle_nulls(df)            — "-" → NaN
   → deduplicate(df)             — drop_duplicates por id_oferta

3. Normalización (normalization.py:19-22)
   → normalize_companies(df)
   → parse_company_name(text)    — strip de sufijos legales

4. Persistencia (persistence.py:48-55)
   → Si empresa es NaN → emp_id = None
   → Si empresa es string → busca/crea Empresa(nombre=...)

5. Modelo DB (models.py:5-8)
   → Empresa: id (PK), nombre (String(255))
   → Sin UNIQUE en nombre, sin nullable=False

6. API (stats.py:67-96)
   → LEFT JOIN empresas e ON o.empresa_id = e.id
   → 5 funciones de métricas sobre df['empresa']
```

---

## Hallazgo 1: Normalización débil — sufijos legales con espacios no se recortan

| Campo | Valor |
|---|---|
| **Archivo** | `analytics/processes/parsing.py` |
| **Función** | `parse_company_name()` |
| **Línea** | 43-48 |
| **Severidad** | Media (afecta calidad de datos, duplica empresas) |
| **Tipo** | Normalización / regex |

### Código actual

```python
COMPANY_SUFFIXES_PATTERN = re.compile(
    r'\b(S\.?A\.?S\.?|L\.?T\.?D\.?A\.?|S\.?A\.?|I\.?N\.?C\.?|B\.?I\.?C\.?)\b',
    re.IGNORECASE
)

def parse_company_name(text):
    if pd.isna(text): 
        return text
    clean = COMPANY_SUFFIXES_PATTERN.sub('', str(text))
    return clean.strip().rstrip('.')
```

### ¿Qué falla?

El regex usa `\b` (word boundary) que exige que el sufijo sea una sola palabra. Pero en la práctica los sufijos aparecen **con espacios entre letras**:

| Entrada scrapeada | Regex espera | ¿Match? | Resultado |
|---|---|---|---|
| `CONATEMPO S.A.S.` | `S.A.S.` (una palabra) | Sí | `CONATEMPO` |
| `CONATEMPO S. A. S` | `S. A. S` (tres palabras) | **No** | `CONATEMPO S. A. S` |
| `ACTIVOS S A S` | `S A S` (tres palabras) | **No** | `ACTIVOS S A S` |
| `EMPRESA LTDA` | `LTDA` | Sí | `EMPRESA` |
| `EMPRESA L T D A` | `L T D A` | **No** | `EMPRESA L T D A` |

### Evidencia en el output real

```json
"top_10_empresas": {
    "CONATEMPO S. A. S": 2,    ← sufijo NO recortado
    "ACTIVOS S A S": 2,         ← sufijo NO recortado
    "Gi Group Colombia": 2,
    "Estrategia Segura": 2
}
```

### Consecuencia

Si la misma empresa aparece en dos ofertas con formato distinto (`S.A.S.` vs `S. A. S`), se crean **dos registros en la tabla `empresas`** con nombres que representan la misma entidad. Esto infla `total_empresas_unicas` y distorsiona `top_10`, `larga_cola`, etc.

### Sugerencia

```python
COMPANY_SUFFIXES_PATTERN = re.compile(
    r'\b(S\.?\s*A\.?\s*S\.?|L\.?\s*T\.?\s*D\.?\s*A\.?|S\.?\s*A\.?|I\.?\s*N\.?\s*C\.?|B\.?\s*I\.?\s*C\.?)\b',
    re.IGNORECASE
)
```

`\s*` entre cada letra captura las variantes con y sin espacios.

---

## Hallazgo 2: `Empresa.nombre` sin UNIQUE — riesgo de duplicados silenciosos

| Campo | Valor |
|---|---|
| **Archivo** | `database/models.py` |
| **Línea** | 8 |
| **Severidad** | Media (riesgo silencioso de duplicados) |
| **Tipo** | Integridad / esquema |

### Código actual

```python
class Empresa(Base):
    __tablename__ = 'empresas'
    id = Column(Integer, primary_key=True, autoincrement=True)
    nombre = Column(String(255))
```

### ¿Qué falla?

`nombre` no tiene `unique=True`. La capa Python (`persistence.py:50`) hace un `filter_by().first()` para evitar duplicados, pero:

1. **Si dos procesos concurrentes** ejecutan `run_dds.py` y `run_fullstack.py` al mismo tiempo, ambos pueden encontrar que `"CONATEMPO"` no existe y ambos insertarlo → 2 filas.
2. **Si la normalización produce variantes** (Hallazgo 1), se insertan como empresas distintas sin que nada lo rechace.

### Sugerencia

```python
nombre = Column(String(255), unique=True, nullable=False)
```

**Nota**: Agregar `unique=True` requiere migración. Si ya hay duplicados (por el Hallazgo 1), la migración fallará. Hay que limpiar primero.

---

## Hallazgo 3: `get_top_n_companies` — `head(10)` incluye empresas con 1 oferta por empates

| Campo | Valor |
|---|---|
| **Archivo** | `mining_stats/metrics.py` |
| **Función** | `get_top_n_companies()` |
| **Línea** | 24-26 |
| **Severidad** | Baja (presentación, no integridad) |
| **Tipo** | Semántica / UX de datos |

### Código actual

```python
def get_top_n_companies(df, n=10):
    return df['empresa'].value_counts().head(n)
```

### ¿Qué falla?

`value_counts()` ordena de mayor a menor. `head(10)` toma **exactamente 10 filas**, sin considerar empates en la posición 10.

En el output real:

```json
"top_10_empresas": {
    "Alianza Temporal": 3,          ← legítimo
    "GENTE UTIL": 2,                ← legítimo
    "ASIGNAR": 2,
    "CONATEMPO S. A. S": 2,
    "Gi Group Colombia": 2,
    "Estrategia Segura": 2,
    "ACTIVOS S A S": 2,
    "Belltech Colombia": 2,
    "INVERSIONES MERIZALDE RESTREPO": 1,  ← empate en count=1
    "software journeys": 1               ← empate en count=1
}
```

Las posiciones 9 y 10 tienen count=1. Pero hay **docenas** de empresas con count=1 (el 84.31% del total). Las que aparecen son arbitrarias (las primeras que pandas encuentra en orden alfabético o de inserción).

### Sugerencia

```python
def get_top_n_companies(df, n=10):
    counts = df['empresa'].value_counts()
    if len(counts) == 0:
        return pd.Series(dtype=int)
    threshold = counts.iloc[min(n-1, len(counts)-1)]
    return counts[counts >= threshold]
```

Esto incluye **todas** las empresas que empatan en el umbral del top N.

---

## Hallazgo 4: `ratio_nulos_empresa: 13.04%` — dato esperado, no bug

| Campo | Valor |
|---|---|
| **Severidad** | Informativo |
| **Tipo** | Calidad de datos |

### ¿De dónde vienen los nulos?

1. El scraper no encuentra el nombre de empresa en el DOM → `get_safe_text` retorna `"-"`
2. `handle_nulls` → `"-"` → `NaN`
3. `persistence.py` → `emp_id = None`
4. `LEFT JOIN` → `empresa = NULL` en el resultado
5. `get_company_integrity` → `isna().mean() * 100 = 13.04%`

### ¿Es correcto?

Sí. El 13% de ofertas sin empresa es un **dato real** del mercado: ofertas de empresas que publican de forma anónima, consultoras que no revelan el cliente, etc. La métrica es fiel al origen.

### ¿Se podría mejorar?

Opcional: diferenciar en el output entre "sin empresa (no scrapeado)" y "empresa confidencial". Pero el scraper actual no tiene forma de distinguirlo — ambos casos producen `"-"`.

---

## Hallazgo 5: `sanitize_text` + `handle_nulls` + `normalize` — orden correcto

| Campo | Valor |
|---|---|
| **Severidad** | Informativo (sin acción requerida) |
| **Tipo** | Buenas prácticas |

El orden en `pipeline.py:14-22`:

```
1. sanitize_text(df)     → limpia Unicode, colapsa espacios
2. handle_nulls(df)      → "-" → NaN
3. deduplicate(df)       → drop_duplicates por id_oferta
4. normalize_companies(df) → strip sufijos legales
```

Es el orden correcto:
- Primero limpiar texto crudo (espacios, tildes, caracteres raros)
- Luego marcar nulos (para no confundir `"-"` válido con centinela `"-"`)
- Luego deduplicar (sobre el id único)
- Luego normalizar (sobre texto ya limpio)

**Nota**: `handle_nulls` usa `df.replace('-', np.nan)` que reemplaza el valor exacto `"-"`, no substrings. Empresas con guiones en el nombre (ej: `"Jean-Marc Soluciones"`) no se ven afectadas.

---

## Hallazgo 6: `get_long_tail_analysis` excluye correctamente los NaN

| Campo | Valor |
|---|---|
| **Severidad** | Informativo (comportamiento correcto) |
| **Tipo** | Verificación |

```python
def get_long_tail_analysis(df):
    counts = df['empresa'].value_counts()
    single_offer_companies = (counts == 1).sum()
    total_companies = len(counts)
    percent = (single_offer_companies / total_companies) * 100 if total_companies > 0 else 0
    return single_offer_companies, percent
```

`value_counts()` por defecto excluye NaN (`dropna=True`). Por tanto:
- `total_companies = 51` → solo empresas con nombre (no incluye los NULL del 13%)
- `single_offer_companies = 43` → correcto
- `porcentaje_larga_cola = 84.31%` → 43/51

Las ofertas sin empresa no distorsionan la métrica de larga cola. Correcto.

---

## Hallazgo 7: `/origen-empresa` — mismo JOIN, mismas vulnerabilidades

| Campo | Valor |
|---|---|
| **Archivo** | `api/routes/stats.py` |
| **Endpoint** | `/origen-empresa` (línea 407) |
| **Severidad** | Baja (misma raíz que Hallazgos 1, 2, 3) |

Usa el mismo `LEFT JOIN empresas` y las mismas operaciones `value_counts().head(3)`, `nunique()`. Hereda:
- Normalización incompleta de sufijos espaciados (H1)
- `head(3)` con empates arbitrarios (H3)
- Riesgo de duplicados por falta de UNIQUE (H2)

No requiere acción separada — se corrige al resolver los hallazgos raíz.

---

## Resumen de hallazgos

| # | Hallazgo | Archivo | Línea | Severidad | Acción |
|---|---|---|---|---|---|
| 1 | Regex de sufijos no captura variantes con espacios (`S. A. S`) | `parsing.py` | 43-48 | Media | Agregar `\s*` entre letras en el patrón |
| 2 | `Empresa.nombre` sin `unique=True` | `models.py` | 8 | Media | Agregar `unique=True, nullable=False` |
| 3 | `get_top_n_companies` incluye empates arbitrarios en `head(10)` | `metrics.py` | 24-26 | Baja | Usar umbral en vez de `head()` fijo |
| 4 | `ratio_nulos: 13.04%` es dato real, no bug | — | — | ✅ OK | Sin acción |
| 5 | Orden de pipeline (sanitize → nulls → dedup → normalize) correcto | `pipeline.py` | 14-22 | ✅ OK | Sin acción |
| 6 | `get_long_tail_analysis` excluye NaN correctamente | `metrics.py` | 41-47 | ✅ OK | Sin acción |
| 7 | `/origen-empresa` hereda vulnerabilidades de H1, H2, H3 | `stats.py` | 407-437 | Baja | Se corrige al resolver H1-H3 |

---

## Datos del output real (testigo)

```json
{
    "metrica": "Empresa",
    "total_empresas_unicas": 51,
    "top_10_empresas": {
        "Alianza Temporal": 3,
        "GENTE UTIL": 2,
        "ASIGNAR": 2,
        "CONATEMPO S. A. S": 2,
        "Gi Group Colombia": 2,
        "Estrategia Segura": 2,
        "ACTIVOS S A S": 2,
        "Belltech Colombia": 2,
        "INVERSIONES MERIZALDE RESTREPO": 1,
        "software journeys": 1
    },
    "ratio_nulos_empresa": "13.04%",
    "total_empresas_solo_ingles": 4,
    "empresas_solo_ingles": [
        "Ayuda Profesional",
        "Grupo Vicca INVERSIONES SUPER ROYAL CARIBE S A",
        "Latin Promo Multimedia",
        "TURISVIVIENDA"
    ],
    "analisis_larga_cola": {
        "empresas_con_una_oferta": 43,
        "porcentaje_larga_cola": "84.31%"
    }
}
```

**Nota**: `"Grupo Vicca INVERSIONES SUPER ROYAL CARIBE S A"` — posible candidato a normalización ("S A" al final).